# ARC-AGI-2 | Task Atlas and Grid Viewer

A compact, competition-input-only atlas for the ARC-AGI-2 training split. It summarizes task scale,
grid shapes, color usage and shape-changing demonstrations, then provides a reusable viewer helper.

Scope: training challenges and training solutions only. This notebook does not read evaluation/test files,
does not create a submission, and does not claim a solver score.

**Verified Kaggle run, 2026-09-07:** 1,000 training tasks, 3,232 demonstration pairs, 1,076 training queries.
1,118 demonstrations change shape; 320 tasks (32%) contain at least one such pair.
Color counts below are cell-weighted across demonstrations, not task-weighted.

Sources: [official competition](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2),
[data](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/data),
[rules](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/rules).
Obtain data through the official input. Raw grids are not exported; the viewer demonstration is synthetic.
Prepared with AI assistance and checked against the official training input.

In [ ]:
import json
import os
import hashlib
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    raise FileNotFoundError('ARC data root not found')

multi_root = os.environ.get('MULTI_DATA_ROOT')
data_root = first_existing([
    os.environ.get('ARC_DATA_ROOT'),
    Path(multi_root) / 'arc2' if multi_root else None,
    Path('/kaggle/input/arc-prize-2026-arc-agi-2'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2'),
    Path('work/multi_competition/data/arc2'),
])
working = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('work/multi_competition/local_runs/arc')
working.mkdir(parents=True, exist_ok=True)

challenges_path = data_root / 'arc-agi_training_challenges.json'
solutions_path = data_root / 'arc-agi_training_solutions.json'
challenges = json.loads(challenges_path.read_text())
solutions = json.loads(solutions_path.read_text())

print(f'Data root: {data_root}')
print(f'Training tasks: {len(challenges):,}')

In [ ]:
def shape(grid):
    return (len(grid), len(grid[0]) if grid else 0)

rows = []
pair_rows = []
color_counts = Counter()
for task_id, task in challenges.items():
    train_pairs = task['train']
    test_inputs = task['test']
    sol_grids = solutions.get(task_id, [])
    task_colors = set()
    shape_changes = 0
    area_ratios = []
    for i, pair in enumerate(train_pairs):
        ishape, oshape = shape(pair['input']), shape(pair['output'])
        if ishape != oshape:
            shape_changes += 1
        in_area = max(1, ishape[0] * ishape[1])
        out_area = oshape[0] * oshape[1]
        area_ratios.append(out_area / in_area)
        vals = [v for row in pair['input'] + pair['output'] for v in row]
        color_counts.update(vals)
        task_colors.update(vals)
        pair_rows.append({
            'task_id_hash': hashlib.sha256(task_id.encode()).hexdigest()[:12],
            'pair_index': i,
            'input_h': ishape[0], 'input_w': ishape[1],
            'output_h': oshape[0], 'output_w': oshape[1],
            'same_shape': ishape == oshape,
            'area_ratio': out_area / in_area,
            'n_colors': len(set(vals)),
        })
    rows.append({
        'n_train_pairs': len(train_pairs),
        'n_test_queries': len(test_inputs),
        'n_solutions_available': len(sol_grids),
        'shape_changing_pairs': shape_changes,
        'any_shape_change': shape_changes > 0,
        'median_area_ratio': float(np.median(area_ratios)),
        'task_color_count': len(task_colors),
        'max_input_area': max(shape(t['input'])[0] * shape(t['input'])[1] for t in train_pairs + test_inputs),
    })

task_df = pd.DataFrame(rows)
pair_df = pd.DataFrame(pair_rows)
summary = {
    'competition': 'arc-prize-2026-arc-agi-2',
    'split': 'training only',
    'task_count': int(len(challenges)),
    'demo_pair_count': int(len(pair_df)),
    'query_count': int(task_df['n_test_queries'].sum()),
    'shape_changing_demo_pairs': int((~pair_df['same_shape']).sum()),
    'tasks_with_any_shape_change': int(task_df['any_shape_change'].sum()),
    'challenge_sha256': hashlib.sha256(challenges_path.read_bytes()).hexdigest(),
    'solutions_sha256': hashlib.sha256(solutions_path.read_bytes()).hexdigest(),
}
print(json.dumps(summary, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
task_df['n_train_pairs'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#34699a')
axes[0].set_title('Training examples per task')
axes[0].set_xlabel('Pairs')
axes[0].set_ylabel('Tasks')

axes[1].scatter(pair_df['input_h'] * pair_df['input_w'], pair_df['output_h'] * pair_df['output_w'],
                alpha=0.25, s=14, color='#b44b32')
lim = max(pair_df['input_h'].mul(pair_df['input_w']).max(), pair_df['output_h'].mul(pair_df['output_w']).max())
axes[1].plot([0, lim], [0, lim], color='black', linewidth=1)
axes[1].set_title('Input area vs output area')
axes[1].set_xlabel('Input cells')
axes[1].set_ylabel('Output cells')

pd.Series(color_counts).sort_index().plot(kind='bar', ax=axes[2], color='#638d3d')
axes[2].set_title('Color token frequency')
axes[2].set_xlabel('Color')
axes[2].set_ylabel('Cells observed')
fig.tight_layout()
fig.savefig(working / 'arc_task_atlas.png', dpi=160)
plt.show()

In [ ]:
aggregates = pd.DataFrame([
    {'metric': 'tasks', 'value': summary['task_count']},
    {'metric': 'demo_pairs', 'value': summary['demo_pair_count']},
    {'metric': 'training_queries', 'value': summary['query_count']},
    {'metric': 'shape_changing_demo_pairs', 'value': summary['shape_changing_demo_pairs']},
    {'metric': 'tasks_with_shape_change', 'value': summary['tasks_with_any_shape_change']},
    {'metric': 'median_train_pairs_per_task', 'value': float(task_df['n_train_pairs'].median())},
    {'metric': 'median_task_color_count', 'value': float(task_df['task_color_count'].median())},
    {'metric': 'p90_max_input_area', 'value': float(task_df['max_input_area'].quantile(0.90))},
])
aggregates.to_csv(working / 'arc_task_aggregates.csv', index=False)
(working / 'summary.json').write_text(json.dumps(summary, indent=2))
aggregates

In [ ]:
ARC_CMAP = ['#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00', '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25']

def show_grid(grid, ax=None, title=None):
    arr = np.array(grid)
    if ax is None:
        _, ax = plt.subplots(figsize=(2.6, 2.6))
    from matplotlib.colors import ListedColormap
    ax.imshow(arr, vmin=0, vmax=9, cmap=ListedColormap(ARC_CMAP), interpolation='nearest')
    ax.set_xticks(np.arange(-.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, arr.shape[0], 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.6)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    if title:
        ax.set_title(title)
    return ax

def show_task(task_id, max_pairs=4):
    task = challenges[task_id]
    pairs = task['train'][:max_pairs]
    fig, axes = plt.subplots(len(pairs), 2, figsize=(5.2, max(2.4, 2.2 * len(pairs))))
    axes = np.atleast_2d(axes)
    for i, pair in enumerate(pairs):
        show_grid(pair['input'], axes[i, 0], f'train {i} input')
        show_grid(pair['output'], axes[i, 1], f'train {i} output')
    fig.tight_layout()
    return fig

# Synthetic demo only: no raw ARC task is displayed by default.
fig, axes = plt.subplots(1, 2, figsize=(4.8, 2.3))
show_grid([[0, 1, 0], [1, 1, 1], [0, 1, 0]], axes[0], 'synthetic input')
show_grid([[1, 1, 1], [1, 0, 1], [1, 1, 1]], axes[1], 'synthetic output')
fig.tight_layout()
plt.show()

## Practical use

The high-value takeaway is that ARC-AGI-2 utility code should treat output shape as a first-class target.
A viewer or solver audit that assumes same-size outputs misses a large fraction of demonstrations.